In [ ]:
#|default_exp resnet

# Load deps

In [ ]:
# !pip install -q torcheval timm
# # or
# !uv add torcheval timm

In [ ]:
# # if src modules imported
# from google.colab import drive
import sys
# drive.mount('/content/drive')
# app_path = '/content/drive/MyDrive/Projects/miniSD'
app_path = "/home/amin/Projects/fast-ai"
sys.path.append(app_path)

In [ ]:
#|export
import torch
import matplotlib as mpl
from functools import partial
import torchvision.transforms.functional as TF,torch.nn.functional as F
from torch import nn,optim
from torch.optim import lr_scheduler
from torcheval.metrics import MulticlassAccuracy
from datasets import load_dataset

from src.utils import set_seed
from src.datasets import inplace, DataLoaders
from src.learner import (
    DeviceCB, MetricsCB, ProgressCB, TrainLearner,
    SingleBatchCB, Learner, MomentumLearner
)
from src.activations import ActivationStats, Hooks
from src.init import GeneralRelu, init_weights, conv
from src.conv import def_device
from src.sgd import BatchSchedCB

# Config

In [ ]:
mpl.rcParams['image.cmap'] = 'gray'
torch.set_printoptions(precision=2, linewidth=140, sci_mode=False)
set_seed(42)
ds_name = "fashion_mnist"
ds_xl,ds_yl = 'image','label'
bs = 1024
num_dl_workers = 4

# Load data

In [ ]:
dsd = load_dataset(ds_name)

train_input_avg = dsd["train"]\
  .map(
      lambda x: {
          "input_mean": [
              TF.to_tensor(image).mean() for image in x[ds_xl]
          ],
          "input_std": [
              TF.to_tensor(image).std() for image in x[ds_xl]
          ]
      },
      batched=True,
      batch_size=10000,
      remove_columns=dsd["train"].column_names,
  )

input_mean_df = train_input_avg.to_pandas().mean()
xmean = input_mean_df["input_mean"]
xstd = input_mean_df["input_std"]

In [ ]:
@inplace
def transformi(b): b[ds_xl] = [(TF.to_tensor(o)-xmean)/xstd for o in b[ds_xl]]

tds = dsd.with_transform(transformi)
dls = DataLoaders.from_dd(tds, bs, num_workers=num_dl_workers)

# Set up callbacks

In [ ]:
#|export
act_gr = partial(GeneralRelu, leak=0.1, sub=0.4)

In [ ]:
metrics = MetricsCB(accuracy=MulticlassAccuracy())
astats = ActivationStats(lambda mod: isinstance(mod, GeneralRelu))
cbs = [DeviceCB(), metrics, ProgressCB(plot=True), astats]
iw = partial(init_weights, leaky=0.1)

# Going deeper

In [ ]:
def get_model(act=nn.ReLU, nfs=(8,16,32,64,128), norm=nn.BatchNorm2d):
    layers = [conv(1, 8, stride=1, act=act, norm=norm)]
    layers += [conv(nfs[i], nfs[i+1], act=act, norm=norm) for i in range(len(nfs)-1)]
    return nn.Sequential(*layers, conv(nfs[-1], 10, act=None, norm=norm, bias=True), nn.Flatten()).to(def_device)

In [ ]:
set_seed(42)
lr,epochs = 6e-2,5
model = get_model(act_gr, norm=nn.BatchNorm2d).apply(iw)
tmax = epochs * len(dls.train)
sched = partial(lr_scheduler.OneCycleLR, max_lr=lr, total_steps=tmax)
xtra = [BatchSchedCB(sched)]
learn = TrainLearner(model, dls, F.cross_entropy, lr=lr, cbs=cbs+xtra, opt_func=optim.AdamW)
learn.fit(epochs)

# Skip Connections

- The ResNet (*residual network*) was introduced in 2015 by Kaiming He et al in the article ["Deep Residual Learning for Image Recognition"](https://arxiv.org/abs/1512.03385).
- The key idea is using a *skip connection* to allow deeper networks to train successfully.

In [ ]:
#|export --> utils
def noop(x): return x

## Define the residual block

In [ ]:
#|export
def _conv_block(ni, nf, stride, act=act_gr, norm=None, ks=3):
    conv1 = conv(ni, nf, stride=1, act=act, norm=norm, ks=ks)
    conv2 = conv(nf, nf, stride=stride, act=None, norm=norm, ks=ks)
    # in order to have actual identity connections at first,
    # we set change initial weights(multiplier) of the BN layer from 1(default) to 0
    # it might only improve the performance for extremely deep networks.
    # TODO: does removing it improve the performance for our network?
    if norm: nn.init.constant_(conv2[1].weight, 0.)
    return nn.Sequential(
        conv1,
        conv2
    )

class ResBlock(nn.Module):
    def __init__(self, ni, nf, stride=1, ks=3, act=act_gr, norm=None):
        super().__init__()
        self.convs = _conv_block(ni, nf, stride, act=act, ks=ks, norm=norm)
        # idconv is used to make input and conv2(conv1(input)) the same shape
        self.idconv = noop if ni==nf else conv(ni, nf, ks=1, stride=1, act=None)
        # TODO: this only works for stride = 1,2. implement the general case
        self.pool = noop if stride==1 else nn.AvgPool2d(2, ceil_mode=True)
        self.act = act()

    def forward(self, x): return self.act(self.convs(x) + self.idconv(self.pool(x)))

## Define the new model arch

In [ ]:
def get_model(act=nn.ReLU, nfs=(8,16,32,64,128,256), norm=nn.BatchNorm2d):
    layers = [ResBlock(1, 8, stride=1, act=act, norm=norm)]
    layers += [ResBlock(nfs[i], nfs[i+1], act=act, norm=norm, stride=2) for i in range(len(nfs)-1)]
    # here we replaced conv --> flatten by flatten --> linear
    # this are equivalent because the last conv is applied on a 1x1(H=W=1) input
    # and the default aggregation over input filters is summation
    layers += [nn.Flatten(), nn.Linear(nfs[-1], 10, bias=False), nn.BatchNorm1d(10)]
    return nn.Sequential(*layers).to(def_device)

## Display model-data dimension DAG

In [ ]:
def _print_shape(hook, mod, inp, outp): print(type(mod).__name__, inp[0].shape, outp.shape)
model = get_model()
learn = TrainLearner(model, dls, F.cross_entropy, cbs=[DeviceCB(), SingleBatchCB()])
with Hooks(model, _print_shape) as hooks: learn.fit(1, train=False)

In [ ]:
#|export --> utils
from IPython import get_ipython

def in_colab():
    "Check if the code is running in Google Colaboratory"
    return 'google.colab' in sys.modules

def ipython_shell():
    "Same as `get_ipython` but returns `False` if not in IPython"
    try: return get_ipython()
    except NameError: return False

def in_ipython():
    "Check if code is running in some kind of IPython environment"
    return bool(ipython_shell())

def in_jupyter():
    "Check if the code is running in a jupyter notebook"
    if not in_ipython(): return False
    return 'InteractiveShell' in ipython_shell().__class__.__name__

def in_notebook():
    "Check if the code is running in a jupyter notebook"
    return in_colab() or in_jupyter()

In [ ]:
def summary(self:Learner):
    res = '|Module|Input|Output|Num params|\n|--|--|--|--|\n'
    tot = 0
    def _f(hook, mod, inp, outp):
        nonlocal res,tot
        nparms = sum(o.numel() for o in mod.parameters())
        tot += nparms
        res += f'|{type(mod).__name__}|{tuple(inp[0].shape)}|{tuple(outp.shape)}|{nparms}|\n'
    with Hooks(self.model, _f) as hooks: self.fit(1, lr=1, train=False, cbs=[SingleBatchCB()])
    print("Tot params: ", tot)
    if in_notebook():
        from IPython.display import Markdown
        return Markdown(res)
    else: print(res)

Learner.summary = summary

In [ ]:
TrainLearner(get_model(), dls, F.cross_entropy, cbs=[DeviceCB()]).summary()

In [ ]:
model = get_model(act_gr, norm=nn.BatchNorm2d).apply(iw)
MomentumLearner(model, dls, F.cross_entropy, lr = 2e-2, cbs=[DeviceCB()]).lr_find()

In [ ]:
lr = 2e-2
tmax = epochs * len(dls.train)
sched = partial(lr_scheduler.OneCycleLR, max_lr=lr, total_steps=tmax)
xtra = [BatchSchedCB(sched)]
model = get_model(act_gr, norm=nn.BatchNorm2d).apply(iw)
learn = TrainLearner(model, dls, F.cross_entropy, lr=lr, cbs=cbs+xtra, opt_func=optim.AdamW)
learn.fit(epochs)